# B2 — Is it really a folding predictor? The structural check

**Question.** If the model has learned folding, its predicted damage should follow **burial** and
ignore the **binding interface**. Structure was never a model input, so this is an independent test.

**What it does.** Places every atlas variant on the 56-position GRB2-SH3 frame, averages the
predicted ΔΔG_fold per position, and correlates it with burial and with distance to the bound
peptide in the solved complex. The measured separation tables from A1 and A2 are shown alongside:
they are the same premise in experimental data.

**Inputs:** B1 atlas predictions, A2 structure, A1/A2 separation tables. **Runtime:** under a minute, CPU.
**Outputs:** `position_structure_check.csv`, `position_predictions.csv`, `RESULTS_B2.md`.
Explanation: `docs/B2_structural_check.md`.

In [1]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'B2', 'Structural check of the folding predictor', resume=RESUME)
print(C.env_info())

Mounted at /content/drive
B2 — Structural check of the folding predictor
  results -> /content/drive/MyDrive/sh3_repro/results/B2_structural_check
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'transformers': '5.16.1', 'torch': '2.11.0+cpu', 'gpu': 'none'}


In [2]:
A = pd.read_parquet(R.res('B1', 'atlas_fold_predictions.parquet'))
STR = pd.read_csv(R.res('A2', 'structure_features.csv'))
ref = STR[STR.domain_key == 'GRB2-SH3'].set_index('pos')
H = A[A.ref_pos.notna()].groupby('ref_pos').agg(n=('pred_ddG_fold', 'size'), mean_fold=('pred_ddG_fold', 'mean')).reset_index()
H['rsasa'] = H.ref_pos.map(ref.rsasa); H['dist_peptide'] = H.ref_pos.map(ref.dist_peptide)
H['buried'] = H.rsasa < 0.25; H['interface'] = H.dist_peptide < 5.0
H.round(4).to_csv(R.out('position_predictions.csv'), index=False)
h = H[H.n >= 10].dropna(subset=['rsasa']).round(4).dropna(subset=['rsasa', 'dist_peptide', 'mean_fold'])
R_BUR = C.spearman(h.mean_fold, 1 - h.rsasa); R_INT = C.spearman(h.mean_fold, -h.dist_peptide)
h.set_index('ref_pos').to_csv(R.out('position_structure_check.csv'))
hv = H.dropna(subset=['rsasa'])
R_ALL = C.spearman(hv.mean_fold, -hv.rsasa)
byclass = h.groupby(['buried', 'interface']).agg(positions=('ref_pos', 'size'), median_pred=('mean_fold', 'median')).reset_index()
var = A.assign(cls=np.where(A.buried.eq(True), 'buried', np.where(A.buried.eq(False), 'surface', 'no structure')))
bv = var.groupby('cls').pred_ddG_fold.agg(['size', 'median']).rename(columns={'size': 'variants', 'median': 'median_pred'}).reset_index()
print(f'{len(h)} positions | burial rho {R_BUR:+.3f} | interface rho {R_INT:+.3f}'); print(byclass.round(3).to_string(index=False))

56 positions | burial rho +0.712 | interface rho +0.037
 buried  interface  positions  median_pred
  False      False         28        0.629
  False       True         12        0.516
   True      False         13        1.404
   True       True          3        1.206


In [3]:
SEP = pd.read_csv(R.res('A1', 'separation_table.csv')); XS = pd.read_csv(R.res('A2', 'cross_fold_separation.csv'))
for k, v in [('positions', len(h)), ('rho_damage_vs_burial', R_BUR), ('rho_damage_vs_interface', R_INT),
             ('rho_damage_vs_burial_all_positions', R_ALL),
             ('burial_to_interface_ratio', abs(R_BUR) / max(abs(R_INT), 1e-9))]:
    R.metric(k, v)
for _, r in bv.iterrows(): R.metric(f'median_pred_{r.cls.replace(" ", "_")}_variants', r.median_pred)
R.section('Predicted damage by structural class (aligned positions with ≥10 variants)', '', byclass, '{:.3f}')
R.section('Predicted damage by the variant’s own structure (AlphaFold)', '', bv, '{:.3f}')
R.section('Measured, for comparison: GRB2-SH3 ddPCA (from A1)', 'Median measured ΔΔG (kcal/mol).', SEP, '{:.3f}')
R.section('Measured, cross-fold replication (from A2)', '', XS, '{:.3f}')
R.finish([f"Predicted damage follows burial: ρ {R_BUR:+.3f} over {len(h)} aligned positions.",
          f"It ignores the binding interface: ρ {R_INT:+.3f} — the burial signal is {abs(R_BUR)/max(abs(R_INT),1e-9):.0f}× larger.",
          'Structure was not a model input, so this is independent evidence that the head learned folding, not binding.'])


B2: REPRODUCED
               quantity  this_run  recorded    status
              positions 56.000000 56.000000 identical
   rho_damage_vs_burial  0.711651  0.711651 identical
rho_damage_vs_interface  0.036979  0.036979 identical

wrote RESULTS_B2.md, metrics_B2.json, comparison_B2.csv


,quantity,this_run,recorded,status
0,positions,56.000000,56.000000,identical
1,rho_damage_vs_burial,0.711651,0.711651,identical
2,rho_damage_vs_interface,0.036979,0.036979,identical
3,rho_damage_vs_burial_all_positions,0.711651,NaN,
4,burial_to_interface_ratio,19.244822,NaN,
5,median_pred_buried_variants,1.161087,NaN,
6,median_pred_no_structure_variants,0.659247,NaN,
7,median_pred_surface_variants,0.377770,NaN,
